# Session 3: a day in the life of the controller
**A. Real-world problem.** At dusk the light does not fall smoothly: clouds, trees and passing cars
make it wobble. A comparator with a single threshold then flips the LED on and off, on and off,
as the noisy light crosses the threshold. That *chatter* wears relays and annoys people. The cure
is a *Schmitt trigger*: two thresholds, so once the LED is on it needs clearly more light to turn
off. Today you simulate 24 hours, count the flips, check the thresholds against hand calculation,
test how component tolerances move the trip point, and build the logic for a two-condition alarm.

In [ ]:
import itertools
import pathlib

import numpy as np

OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"
VCC = 5.0          # V
R10 = 10_000.0     # ohm, assumed LDR resistance at 10 lux (illustrative)
GAMMA = 0.7        # assumed LDR slope (illustrative)
R_FIXED = 6800.0   # ohm, divider resistor from Session 2
R_LED = 330.0      # ohm, LED resistor from Session 2
VF = 2.0           # V, typical red LED forward voltage (typical)
VCE_SAT = 0.2      # V, typical saturated transistor (typical)
R_BASE = 4700.0    # ohm, base resistor from Session 2
VBE = 0.7          # V, typical
V_LO, V_HI = 2.25, 2.75      # V, Schmitt thresholds: LED on below V_LO, off above V_HI
DT_MIN = 1.0       # minutes per simulation step

## Step 1: the sensor and the hand-calculated thresholds
Same maths as Session 2: V = Vcc R / (R + R_ldr), and the light level at which V equals a threshold
is lux = 10 (R10 / R_ldr) ** (1 / gamma) with R_ldr = R (Vcc - V) / V.

In [ ]:
def ldr_resistance(lux, r10=R10, gamma=GAMMA):
    return r10 * (np.asarray(lux, dtype=float) / 10.0) ** (-gamma)


def sensor_voltage(lux, vcc=VCC, r=R_FIXED, r10=R10, gamma=GAMMA):
    r_ldr = ldr_resistance(lux, r10, gamma)
    return vcc * r / (r + r_ldr)


def lux_at_voltage(v, vcc=VCC, r=R_FIXED, r10=R10, gamma=GAMMA):
    """Light level (lux) at which the divider output is v: the hand-calculated threshold."""
    r_ldr = r * (vcc - v) / v
    return 10.0 * (r10 / r_ldr) ** (1.0 / gamma)

## Step 2: a synthetic day
The log of the light level follows a sine over 24 hours: log10(lux) = 1.3 + 3.2 sin(2 pi (t - 6) / 24).
It crosses 20 lux at 06:00 and 18:00, peaks near 31 600 lux at noon and falls to 0.013 lux at midnight.
Clouds are a slowly varying random factor exp(sigma * n), where n is a smoothed random number
(AR(1), seeded so every run gives the same day).

In [ ]:
def clean_lux(minutes):
    t = np.asarray(minutes, dtype=float) / 60.0
    return 10.0 ** (1.3 + 3.2 * np.sin(2 * np.pi * (t - 6.0) / 24.0))


def cloud_factor(n, seed, sigma=0.35, rho=0.8):
    """Seeded, smoothly varying multiplicative noise with a log-standard-deviation sigma."""
    rng = np.random.default_rng(seed)
    e = rng.standard_normal(n)
    x = np.empty(n)
    x[0] = e[0]
    k = np.sqrt(1.0 - rho ** 2)
    for i in range(1, n):
        x[i] = rho * x[i - 1] + k * e[i]
    return np.exp(sigma * x)


def day_lux(seed=7, sigma=0.35, n=1440):
    minutes = np.arange(n) * DT_MIN
    lux = clean_lux(minutes)
    return minutes, lux * cloud_factor(n, seed, sigma) if sigma > 0 else lux

## Step 3: two kinds of comparator
* plain: the LED is on whenever V is below one threshold V_th = (V_LO + V_HI) / 2
* Schmitt: on when V drops below V_LO; off only when V rises above V_HI; in between it keeps its state.

In [ ]:
def plain_comparator(v, v_th=(V_LO + V_HI) / 2):
    return np.asarray(v) < v_th


def schmitt(v, v_lo=V_LO, v_hi=V_HI):
    v = np.asarray(v)
    state = np.empty(len(v), dtype=bool)
    cur = bool(v[0] < (v_lo + v_hi) / 2)
    for i, x in enumerate(v):
        if x < v_lo:
            cur = True
        elif x > v_hi:
            cur = False
        state[i] = cur
    return state


def count_switches(state):
    """Number of state changes (on to off or off to on)."""
    s = np.asarray(state, dtype=int)
    return int(np.abs(np.diff(s)).sum())

## Step 4: power and on-hours
When the LED is on the supply delivers the LED current plus the base current; always, the divider
draws Vcc / (R + R_ldr). Energy = power times time.

In [ ]:
def led_path_current(vcc=VCC, r_led=R_LED, vf=VF, vce_sat=VCE_SAT):
    return (vcc - vf - vce_sat) / r_led


def simulate_day(seed=7, sigma=0.35, hysteresis=True, force_on=None, force_off=None, vcc=VCC):
    """Run the controller for 24 hours. force_on / force_off: optional boolean arrays (manual override)."""
    minutes, lux = day_lux(seed, sigma)
    v = sensor_voltage(lux, vcc)
    dark = schmitt(v) if hysteresis else plain_comparator(v)
    led = dark.copy()
    if force_on is not None:
        led = led | np.asarray(force_on, dtype=bool)
    if force_off is not None:
        led = led & ~np.asarray(force_off, dtype=bool)
    i_led = led_path_current(vcc)
    i_base = (vcc - VBE) / R_BASE
    i_div = vcc / (R_FIXED + ldr_resistance(lux))
    i_supply = i_div + led * (i_led + i_base)
    hours = DT_MIN / 60.0
    return {"minutes": minutes, "lux": lux, "v": v, "dark": dark, "led": led,
            "switches": count_switches(led), "on_hours": float(led.sum() * hours),
            "energy_wh": float(np.sum(vcc * i_supply) * hours), "i_supply": i_supply}

## Step 5: validate against the hand calculation
With no clouds the light is the clean sine, so the switch-on and switch-off times can be calculated
by solving log10(lux) = 1.3 + 3.2 sin(x) for the threshold lux. The simulation must agree to within
one time step.

In [ ]:
def analytic_cross_times(lux_threshold):
    """(falling-light time, rising-light time) in hours when the clean light equals lux_threshold."""
    s = (np.log10(lux_threshold) - 1.3) / 3.2
    x = np.arcsin(s)
    t_fall = 6.0 + 24.0 * (np.pi - x) / (2 * np.pi)       # evening: light falling through the level
    t_rise = 6.0 + 24.0 * x / (2 * np.pi)                  # morning: light rising through the level
    return float(t_fall % 24.0), float(t_rise % 24.0)

## Step 6: Monte Carlo on tolerances
Real parts are not exact. We draw the LDR scale R10 (assumed 20 percent standard deviation, illustrative),
the fixed resistor (up to 5 percent either way) and the supply (4.5 to 5.5 V) and look at the trip light
level. Two designs: a FIXED 2.5 V reference, or a reference that is a fixed fraction of the supply
(ratiometric, Vcc / 2), for which the supply cancels.

In [ ]:
def monte_carlo_trip(n=4000, seed=3, ratiometric=False, r10_sigma=0.20, r_tol=0.05, v_lo=4.5, v_hi=5.5):
    rng = np.random.default_rng(seed)
    r10 = R10 * (1 + r10_sigma * rng.standard_normal(n))
    r = R_FIXED * (1 + r_tol * (2 * rng.random(n) - 1))
    vcc = v_lo + (v_hi - v_lo) * rng.random(n)
    v_th = vcc / 2 if ratiometric else np.full(n, 2.5)
    r_ldr = r * (vcc - v_th) / v_th
    lux = 10.0 * (r10 / r_ldr) ** (1.0 / GAMMA)
    return lux, vcc

## Step 7: a small gate simulator and the two-condition alarm
Digital logic: signals are 0 or 1 and gates combine them. An alarm that fires when EITHER of two
independent safety conditions is violated is OR. Build it from gates and test every input row.

In [ ]:
GATES = {"AND": lambda a, b: a & b, "OR": lambda a, b: a | b, "NAND": lambda a, b: 1 - (a & b),
         "NOR": lambda a, b: 1 - (a | b), "XOR": lambda a, b: a ^ b, "NOT": lambda a: 1 - a}


def evaluate(netlist, inputs):
    """netlist: list of (output_name, gate, [input names]) in order. Returns all signal values."""
    sig = dict(inputs)
    for out, gate, ins in netlist:
        sig[out] = int(GATES[gate](*[sig[i] for i in ins]))
    return sig


def truth_table(netlist, input_names, output_name):
    """All 2**n rows as (input tuple, output) in counting order."""
    rows = []
    for combo in itertools.product((0, 1), repeat=len(input_names)):
        sig = evaluate(netlist, dict(zip(input_names, combo)))
        rows.append((combo, sig[output_name]))
    return rows


ALARM_OR = [("alarm", "OR", ["A", "B"])]
ALARM_NAND = [("nA", "NOT", ["A"]), ("nB", "NOT", ["B"]), ("alarm", "NAND", ["nA", "nB"])]    # De Morgan
ALARM_XOR = [("alarm", "XOR", ["A", "B"])]                                              # the wrong gate
OVERRIDE = [("keep_on", "OR", ["DARK", "FORCE_ON"]), ("n_off", "NOT", ["FORCE_OFF"]), ("led", "AND", ["keep_on", "n_off"])]

## Step 8: the failure cases
(a) no hysteresis: count the flips. (b) the transistor is not saturated: dim LED and extra heat.
For a 100 mA lamp (R = 30 ohm) the transistor is the hot part when half on.

In [ ]:
def unsaturated_power(r_b, r_led, beta=100.0, vcc=VCC):
    i_b = (vcc - VBE) / r_b
    i_sat = (vcc - VF - VCE_SAT) / r_led
    i_c = min(beta * i_b, i_sat)
    vce = VCE_SAT if beta * i_b >= i_sat else vcc - VF - i_c * r_led
    return i_c, vce, i_c * vce

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    lux_on, lux_off = lux_at_voltage(V_LO), lux_at_voltage(V_HI)
    print(f"hand calculation: LED on below {lux_on:.1f} lux ({V_LO} V), off above {lux_off:.1f} lux ({V_HI} V)")
    for hyst in (False, True):
        r = simulate_day(hysteresis=hyst)
        print(f"hysteresis={hyst}: {r['switches']} switches, LED on {r['on_hours']:.2f} h, energy {r['energy_wh'] * 1000:.0f} mWh")
    clean = simulate_day(sigma=0.0, hysteresis=True)
    t_on = analytic_cross_times(lux_on)[0]
    t_off = analytic_cross_times(lux_off)[1]
    print(f"clean day: analytic on at {t_on:.3f} h, off at {t_off:.3f} h, on-hours {(24 - t_on) + t_off:.3f}; "
          f"simulated on-hours {clean['on_hours']:.3f}")
    for ratio in (False, True):
        lux_mc, vcc_mc = monte_carlo_trip(ratiometric=ratio)
        p5, p50, p95 = np.percentile(lux_mc, [5, 50, 95])
        print(f"ratiometric={ratio}: trip lux 5/50/95 percentile = {p5:.1f} / {p50:.1f} / {p95:.1f}")
    for rb in (4700.0, 47000.0, 470000.0):
        ic, vce, pq = unsaturated_power(rb, 30.0)
        print(f"100 mA lamp, R_B = {rb:.0f}: I_C = {ic * 1e3:.1f} mA, V_CE = {vce:.2f} V, P = {pq * 1e3:.1f} mW")

    fig, axes = plt.subplots(3, 1, figsize=(8, 7), sharex=True)
    for hyst, ax, name in ((False, axes[1], "one threshold"), (True, axes[2], "Schmitt trigger")):
        r = simulate_day(hysteresis=hyst)
        ax.step(r["minutes"] / 60, r["led"], where="post")
        ax.set_ylabel("LED (1 = on)")
        ax.set_title(f"{name}: {r['switches']} switches in 24 h")
        ax.grid(alpha=0.3)
    r = simulate_day(hysteresis=True)
    axes[0].semilogy(r["minutes"] / 60, r["lux"], lw=0.8)
    axes[0].axhline(lux_on, color="green", ls="--", label=f"on below {lux_on:.1f} lux")
    axes[0].axhline(lux_off, color="red", ls="--", label=f"off above {lux_off:.1f} lux")
    axes[0].set_ylabel("light (lux)")
    axes[0].set_title("A cloudy synthetic day")
    axes[0].legend(fontsize=8)
    axes[0].grid(alpha=0.3)
    axes[2].set_xlabel("time of day (hours)")
    fig.tight_layout()
    fig.savefig(OUT / "day_simulation.png", dpi=120)

    fig2, ax2 = plt.subplots(figsize=(6.5, 4))
    for ratio, label in ((False, "fixed 2.5 V reference"), (True, "reference = Vcc / 2")):
        lux_mc, _ = monte_carlo_trip(ratiometric=ratio)
        ax2.hist(np.log10(lux_mc), bins=60, alpha=0.55, label=label)
    ax2.set_xlabel("trip point, log10 of light level (lux)")
    ax2.set_ylabel("number of circuits (out of 4000)")
    ax2.set_title("Tolerance spread of the trip point")
    ax2.legend(fontsize=8)
    ax2.grid(alpha=0.3)
    fig2.tight_layout()
    fig2.savefig(OUT / "monte_carlo.png", dpi=120)
    print("saved figures in", OUT)

## Try it
Do the coding tasks, answer the four questions in the README (section F), then run the checkpoint.

## Coding tasks
1. **Easy.** Run `simulate_day(seed=3, hysteresis=False)` and `simulate_day(seed=3)` and compare the switch counts. Then try sigma = 0.7 and say what happens to the Schmitt count.
2. **Stretch.** Write a `truth_table` call for the three-input `OVERRIDE` netlist and check by hand the row DARK = 0, FORCE_ON = 1, FORCE_OFF = 1.
3. **Explain.** Explain why the Monte Carlo spread for the fixed 2.5 V reference is wider than for the Vcc / 2 reference when only the supply varies, and why it is almost equal when all tolerances vary.

## Assessment
- **A. Conceptual.** Why does hysteresis stop chatter without smoothing the light signal? And why is XOR the wrong gate for an alarm that must fire when either of two conditions is violated?
- **B. Numerical.** A wider Schmitt band uses V_LO = 2.0 V and V_HI = 3.0 V with R = 6.8 kilo-ohm. Find the light levels for on and off. Compare the ratio of off-level to on-level with that of the 2.25 V and 2.75 V pair.
- **C. Graph and simulation interpretation.** In the day-simulation figure, why do all the extra switches of the single-threshold comparator cluster near dawn and dusk and none occur at noon or midnight?
- **D. Design and optimisation challenge.** Design: a machine has two independent safety conditions, A (temperature too high) and B (pressure too high). The alarm must fire when either is violated. Write the truth table, choose the gate, and say how you would make a broken sensor wire fail safe. Then add a manual-override input for a maintenance person who may silence the alarm.

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import itertools

import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    # hand-calculated thresholds
    lo, hi = L.lux_at_voltage(2.25), L.lux_at_voltage(2.75)
    checks.append((f"hand calculation: 2.25 V is 13.0 lux and 2.75 V is 23.1 lux (got {lo:.1f}, {hi:.1f})",
                   abs(lo - 13.0) < 0.05 and abs(hi - 23.1) < 0.05))
    checks.append(("the divider at the threshold light levels returns the threshold voltages",
                   abs(float(L.sensor_voltage(lo)) - 2.25) < 1e-9 and abs(float(L.sensor_voltage(hi)) - 2.75) < 1e-9))

    # schmitt behaviour
    v = np.array([3.0, 2.6, 2.3, 2.2, 2.4, 2.7, 2.8, 2.6, 2.3])
    checks.append(("Schmitt: on below 2.25 V, holds between 2.25 and 2.75 V, off above 2.75 V",
                   list(L.schmitt(v)) == [False, False, False, True, True, True, False, False, False]))
    checks.append(("a plain comparator at 2.5 V flips at every crossing, the Schmitt does not",
                   L.count_switches(L.plain_comparator(np.array([2.6, 2.4, 2.6, 2.4, 2.6]))) == 4
                   and L.count_switches(L.schmitt(np.array([2.6, 2.4, 2.6, 2.4, 2.6]))) == 0))
    checks.append(("a Schmitt with equal thresholds behaves like a plain comparator",
                   list(L.schmitt(v, 2.5, 2.5)) == list(L.plain_comparator(v, 2.5))))

    # chatter
    plain = L.simulate_day(seed=7, hysteresis=False)
    hyst = L.simulate_day(seed=7, hysteresis=True)
    checks.append((f"worked example: seed 7 day gives {plain['switches']} switches without hysteresis and {hyst['switches']} with",
                   plain["switches"] == 20 and hyst["switches"] == 2))
    many = all(L.simulate_day(seed=s, hysteresis=False)["switches"] > L.simulate_day(seed=s)["switches"] for s in range(1, 9))
    checks.append(("for 8 different cloudy days hysteresis always gives fewer switches", many))
    checks.append(("a clean (no cloud) day has exactly 2 switches even without hysteresis",
                   L.simulate_day(sigma=0.0, hysteresis=False)["switches"] == 2))

    # validation against the analytic crossing times
    clean = L.simulate_day(sigma=0.0, hysteresis=True)
    t_on = L.analytic_cross_times(lo)[0]
    t_off = L.analytic_cross_times(hi)[1]
    analytic_hours = (24.0 - t_on) + t_off
    checks.append((f"simulated on-hours {clean['on_hours']:.3f} match the analytic {analytic_hours:.3f} to one step",
                   abs(clean["on_hours"] - analytic_hours) < 2 * L.DT_MIN / 60))
    first_on = clean["minutes"][np.argmax(clean["led"] & (clean["minutes"] > 720))] / 60
    checks.append((f"first evening switch-on is at {first_on:.2f} h, analytic {t_on:.2f} h",
                   abs(first_on - t_on) < 2 * L.DT_MIN / 60))
    checks.append(("clean light crosses 20 lux at about 06:00 and 18:00", np.allclose(L.analytic_cross_times(20.0), [18.0, 6.0], atol=0.01)))

    # power and energy
    i_led = L.led_path_current()
    checks.append(("LED path current is (5 - 2 - 0.2) / 330 = 8.48 mA", abs(i_led * 1e3 - 8.485) < 0.01))
    e_expected = hyst["on_hours"] * 5.0 * (i_led + (5.0 - 0.7) / 4700.0)
    checks.append(("energy is at least the LED on-hours times the LED supply power",
                   hyst["energy_wh"] >= e_expected and hyst["energy_wh"] < e_expected + 0.05))

    # Monte Carlo
    fixed, _ = L.monte_carlo_trip(ratiometric=False, r10_sigma=0.0, r_tol=0.0)
    ratio, _ = L.monte_carlo_trip(ratiometric=True, r10_sigma=0.0, r_tol=0.0)
    checks.append(("supply only: a Vcc / 2 reference gives one trip point (17.35 lux), a fixed 2.5 V reference spreads it",
                   np.ptp(ratio) < 1e-9 and abs(ratio[0] - 17.35) < 0.05 and np.ptp(fixed) > 5.0))
    allp, _ = L.monte_carlo_trip(ratiometric=True)
    p5, p50, p95 = np.percentile(allp, [5, 50, 95])
    checks.append((f"with all tolerances the trip point is 9.8 to 26.2 lux (5th-95th percentile), median 17.4 (got {p5:.1f}, {p95:.1f}, {p50:.1f})",
                   abs(p5 - 9.8) < 0.5 and abs(p95 - 26.2) < 1.0 and abs(p50 - 17.4) < 0.5))

    # logic
    names = ["A", "B"]
    expect_or = [((0, 0), 0), ((0, 1), 1), ((1, 0), 1), ((1, 1), 1)]
    checks.append(("worked example: the alarm OR gate gives the truth table 0, 1, 1, 1", L.truth_table(L.ALARM_OR, names, "alarm") == expect_or))
    checks.append(("NOT A NAND NOT B equals OR on every row (De Morgan)",
                   L.truth_table(L.ALARM_NAND, names, "alarm") == expect_or))
    checks.append(("XOR is the wrong alarm: it stays silent when both conditions are violated",
                   L.truth_table(L.ALARM_XOR, names, "alarm")[3][1] == 0))
    for g, f in (("AND", lambda a, b: a and b), ("OR", lambda a, b: a or b), ("NAND", lambda a, b: not (a and b)), ("NOR", lambda a, b: not (a or b))):
        ok = all(L.GATES[g](a, b) == int(f(a, b)) for a, b in itertools.product((0, 1), repeat=2))
        checks.append((f"gate {g} matches Python logic on all 4 rows", ok))
    ok = all(L.truth_table(L.OVERRIDE, ["DARK", "FORCE_ON", "FORCE_OFF"], "led")[i][1] ==
             int((d or fo) and not ff) for i, (d, fo, ff) in enumerate(itertools.product((0, 1), repeat=3)))
    checks.append(("override controller: LED = (DARK or FORCE_ON) and not FORCE_OFF on all 8 rows", ok))

    # override day
    n = 1440
    fo = np.zeros(n, bool)
    fo[720:780] = True
    ff = np.zeros(n, bool)
    ff[1320:1410] = True
    day = L.simulate_day(seed=7, force_on=fo, force_off=ff)
    checks.append((f"override day: 12:00-13:00 on, 22:00-23:30 off gives {day['on_hours']:.2f} h on (baseline {hyst['on_hours']:.2f})",
                   abs(day["on_hours"] - 11.53) < 0.02 and abs(hyst["on_hours"] - 12.03) < 0.02 and day["switches"] == 6))
    both = L.simulate_day(seed=7, force_on=np.ones(n, bool), force_off=np.ones(n, bool))
    checks.append(("edge: if both override buttons are held, the LED is off (off wins)", both["on_hours"] == 0.0))

    # failure case: not saturated
    peak = max(L.unsaturated_power(rb, 30.0)[2] for rb in np.linspace(2000, 200000, 2000))
    sat = L.unsaturated_power(2200.0, 30.0)[2]
    checks.append((f"failure: a half-on transistor for a 100 mA lamp dissipates {peak * 1e3:.0f} mW, over 3 times the saturated {sat * 1e3:.0f} mW",
                   peak > 3 * sat and abs(peak * 1e3 - 75.0) < 1.0))
    ic, vce, _ = L.unsaturated_power(470e3, 330.0)
    checks.append(("failure: R_B = 470 kohm gives under 1 mA in the LED (dim)", ic < 0.001))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
